# 실습 4 군집 모델 구현하기 (학생용)

교과서 120~122쪽

**문제 상황**: 쇼핑몰 고객 정보를 k-평균 군집 알고리즘으로 분석하면 고객 특성에 맞는 마케팅을 할 수 있다. 쇼핑몰 고객을 군집화해 보자.

| 단계 | 하는 일 |
|---|---|
| 단계 1 문제 정의하기 | 쇼핑몰 고객 마케팅을 위해 군집화하기 |
| 단계 2 데이터 수집 및 전처리하기 | Mall_Customers.csv에서 연 소득, 쇼핑몰 점수만 고르기 |
| 단계 3 모델 생성하기 | k-평균 군집 모델 학습, 군집 결과와 중심점 그리기 |
| 단계 4 모델 평가하기 | 실루엣 점수, 엘보 그래프 |

**데이터**: 교과서 120쪽 링크(캐글)의 `Mall_Customers.csv` (고객 200명)

**사용 방법**: 위에서부터 차례로 ▶ 버튼을 눌러 실행합니다. 오른쪽의 선택 상자와 슬라이더로 설정을 바꿔 다시 실행해 보세요.

In [ ]:
#@title 데이터 준비하기
how = "사이트에서 바로 받기" #@param ["사이트에서 바로 받기", "내 컴퓨터에서 올리기"]
#@markdown 디지털 교과서에 올려 둔 교과서 자료(Mall_Customers.csv)를 코랩으로 바로 가져옵니다. 직접 모은 파일을 쓰려면 '내 컴퓨터에서 올리기'를 고르세요.
if how.startswith('사이트'):
    import urllib.request
    for name in ['Mall_Customers.csv']:
        urllib.request.urlretrieve('https://aibook-dl.vercel.app/data/' + name, name)
        print('받았습니다:', name)
else:
    from google.colab import files
    uploaded = files.upload()


In [ ]:
#@title 단계 2 데이터 불러오고 속성 고르기 (교과서 120~121쪽)
import pandas as pd
df = pd.read_csv('Mall_Customers.csv')
data = df[['Annual Income (k$)', 'Spending Score (1-100)']]   # 연 소득, 쇼핑몰 점수
print(df.shape)
data.head()

In [ ]:
#@title 단계 3 k-평균 군집 모델 만들고 그리기 (교과서 121~122쪽)
k = 5 #@param {type:"slider", min:2, max:10, step:1}
from sklearn.cluster import KMeans
import seaborn as sns
import matplotlib.pyplot as plt

model = KMeans(n_clusters=k, n_init=10, random_state=42)   # k개 군집 모델 생성
model.fit(data)                                              # 군집 모델 학습 (타깃 없음)
df['centroid'] = model.labels_                               # 군집 번호 저장
final_centroid = model.cluster_centers_                      # 군집 중심점
print('군집 중심점 (연 소득, 쇼핑몰 점수)'); print(final_centroid.round(1))

sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', hue='centroid', data=df, palette='bright')
plt.scatter(final_centroid[:, 0], final_centroid[:, 1], marker='*', s=300, color='black', label='Centroids')
plt.legend(); plt.show()

In [ ]:
#@title 단계 4 모델 평가하기: 실루엣 점수 (교과서 122쪽)
from sklearn.metrics import silhouette_score
score = silhouette_score(data, model.labels_)
print(f'k = {k}일 때 실루엣 점수: {score:.4f}  (1에 가까울수록 군집이 잘 나뉨)')

In [ ]:
#@title 가장 알맞은 k 찾기: 엘보 그래프와 실루엣 점수 (교과서 120쪽 지식 더하기)
ks = range(2, 11)
sse, sil = [], []
for kk in ks:
    m = KMeans(n_clusters=kk, n_init=10, random_state=42).fit(data)
    sse.append(m.inertia_)                         # 오차 제곱합(SSE)
    sil.append(silhouette_score(data, m.labels_))  # 실루엣 점수
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ks, sse, 'o-'); ax[0].set_title('Elbow (SSE)'); ax[0].set_xlabel('k')
ax[1].plot(ks, sil, 'o-', color='green'); ax[1].set_title('Silhouette'); ax[1].set_xlabel('k')
plt.show()
for kk, s in zip(ks, sil):
    print(f'k={kk}: 실루엣 {s:.4f}')

### 정리하기
1. 실루엣 점수가 가장 높은 k는 무엇인가요? 엘보 그래프에서 꺾이는 곳과 같은가요?
2. k=5일 때 다섯 군집은 각각 어떤 고객들인가요? 군집마다 어울리는 마케팅을 하나씩 정해 보세요.